# Task 3 — CycleGAN Monet ↔ Photo (Nikhil)

Executed record of the full run. Domain **A = Monet**, **B = Photo**; `A2B` = Monet→Photo, `B2A` = Photo→Monet (Kaggle direction).

| File | Contents |
|---|---|
| `data.py` | unpaired loader (random B partner per A), resize → random crop 128 → flip, fixed photo hold-out |
| `models.py` | ResNet generator (6 residual blocks, 64 base channels), 70×70 PatchGAN discriminator, image history pool — all trained from scratch |
| `train.py` | LSGAN + cycle (λ=10) + identity (λ=5) losses, Adam 2e-4 β=(0.5, 0.999), 25 constant + 25 linear-decay epochs, gradient-norm / NaN tracking, checkpoints every 5 epochs, `--resume` |
| `evaluate.py` | FID, KID, precision/recall + density/coverage, cycle L1, LPIPS, content cosine (both directions), blinded 30-sample human-audit sheet, Kaggle `images.zip` |
| `audit_agreement.py` | mean human scores + Cohen's κ / % agreement once both raters fill their sheets |
| `configs/cyclegan_nikhil.yaml` | every hyperparameter for this run |

Pretrained InceptionV3 / AlexNet are used **only inside `evaluate.py` as measuring instruments**; they never touch training or the submitted images.

Data: see `task3_gan/data/README.md`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

SRC = Path.cwd()
assert (SRC / "train.py").exists(), "run this notebook from task3_gan/member_nikhil/src"
sys.path.insert(0, str(SRC))
import train
CONFIG = SRC / "configs" / "cyclegan_nikhil.yaml"
RESUME = None   # e.g. SRC.parent / "checkpoints" / "<run_id>_epoch025.pt" if a session was cut off
print(CONFIG.read_text())

## Train (50 epochs) + evaluate + build Kaggle submission

In [ ]:
result = train.run(CONFIG, resume=RESUME)
out = result["out_dir"]
print(result["run_id"])

## Loss curves and training stability

In [ ]:
display(Image(out / "loss_curves.png"))
pd.read_csv(out / "train_history.csv").round(4)

## Full metrics (both directions)

In [ ]:
pd.set_option("display.max_rows", 100)
pd.read_csv(out / "full_metrics_report.csv")

## Sample translations over training

Rows: real Monet → fake Photo → reconstructed Monet → real Photo → fake Monet → reconstructed Photo (fixed held-out images).

In [ ]:
samples = sorted((out / "samples").glob("epoch*.png"))
for p in [samples[0], samples[len(samples) // 2], samples[-1]]:
    print(p.name); display(Image(p))

## Human audit

Blinded sheets are in `human_audit/` (images numbered, direction hidden in the answer key). Two raters fill `ratings_rater1.csv` / `ratings_rater2.csv`, then run:

```bash
python task3_gan/member_nikhil/src/audit_agreement.py task3_gan/member_nikhil/outputs/<run_id>/human_audit
```